# Phase 2: Lightweight EfficientNet-B0 & MobileNetV3 E-Waste Classifier
### Domain: Computer Vision / Transfer Learning
**SDG Alignment:** SDG 11 (Sustainable Cities) & SDG 12 (Responsible Consumption)

---

## 📌 Hyperparameter Summary (One-Line Explanations)
- `IMAGE_SIZE = 224`: Standard input resolution balancing spatial detail against compute overhead.
- `BATCH_SIZE = 32`: Number of samples per weight update step; fits comfortably in Colab free T4 16GB GPU.
- `STAGE1_LR = 1e-3`: High learning rate to quickly initialize the randomly initialized linear classification head.
- `STAGE1_EPOCHS = 5`: Number of epochs to train only the head before exposing backbone weights.
- `STAGE2_LR = 1e-4`: Small learning rate to gently adapt deep backbone features without catastrophic forgetting.
- `STAGE2_EPOCHS = 10`: Fine-tuning epochs with learning rate decay.
- `WEIGHT_DECAY = 1e-4`: L2 weight regularization preventing individual weight magnitudes from exploding.
- `LABEL_SMOOTHING = 0.1`: Softens one-hot target probabilities (0.9 vs 0.1/(K-1)), curbing model overconfidence.
- `EARLY_STOP_PATIENCE = 5`: Halts training if validation loss fails to improve for 5 consecutive epochs.
- `USE_AMP = True`: Automatic Mixed Precision (FP16/FP32), doubling training speed on Tensor Core GPUs while cutting VRAM in half.

---

## ⚖️ Overfitting & Underfitting Guide
- **Underfitting:** High training loss, high validation loss, low accuracy on both.  
  *Fix:* Unfreeze more backbone layers, increase training epochs, raise learning rate, reduce regularization.
- **Overfitting:** Training loss continues dropping, but validation loss starts rising or stagnates.  
  *Fix:* Apply label smoothing (0.1), add data augmentation (rotation/color jitter/flips), increase weight decay, enable early stopping.


---
## Cell (a): Setup, Seed Fixing, GPU Check, & Drive Mount
Installs dependencies, mounts Google Drive to preserve checkpoints across session disconnects, and fixes the random seed (42).


In [ ]:
# CELL (a): Setup, Seeds, GPU, & Mount Google Drive
import os
import sys
import time
import random
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Install timm if running in Google Colab
try:
    import timm
except ImportError:
    !pip install -q timm==1.0.9
    import timm

# Fix all random seeds for exact reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[STATUS] Training Device: {device}')
if torch.cuda.is_available():
    print(f'[STATUS] GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'[STATUS] VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('[WARN] Running on CPU! Enable GPU in Colab via Runtime -> Change runtime type -> T4 GPU.')

# Mount Google Drive if in Colab
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/ewaste_checkpoints')
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'[STATUS] Checkpoints will be saved to Google Drive: {CHECKPOINT_DIR}')
else:
    CHECKPOINT_DIR = Path('./models/checkpoints')
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'[STATUS] Local checkpoint directory: {CHECKPOINT_DIR}')


---
## Cell (b): Image Transforms & DataLoaders
- **Resolution:** 224x224 (EfficientNet standard)
- **Train split:** Random horizontal flip (p=0.5), random rotation (15 deg), color jitter (0.2), ImageNet normalization.
- **Validation & Test splits:** Deterministic resize to 224x224 + ImageNet normalization (NO augmentation).


In [ ]:
# CELL (b): Image Transforms & DataLoaders
IMAGE_SIZE = 224
BATCH_SIZE = 32
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# In Colab, you can upload data/processed.zip or pull from repo
DATA_DIR = Path('./data/processed')
if not (DATA_DIR / 'train').exists():
    print('[ACTION NEEDED] Place data/processed folder or extract zip into ./data/processed')

train_dataset = ImageFolder(root=DATA_DIR / 'train', transform=train_transforms)
val_dataset   = ImageFolder(root=DATA_DIR / 'val',   transform=val_transforms)
test_dataset  = ImageFolder(root=DATA_DIR / 'test',  transform=val_transforms)

CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'[OK] Discovered {NUM_CLASSES} classes: {CLASS_NAMES}')
print(f'[OK] Split counts -> Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

num_workers = 2 if not sys.platform.startswith('win') else 0
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)


---
## Cell (c): Model Architecture (EfficientNet-B0 & MobileNetV3)
Loads ImageNet pretrained weights via `timm` and replaces the linear head with a 7-class output layer.


In [ ]:
# CELL (c): Model Architecture Builder
def build_model(model_name: str = 'efficientnet_b0', num_classes: int = 7, dropout: float = 0.2):
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=num_classes,
        drop_rate=dropout
    )
    return model

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# Test instantiation
sample_model = build_model('efficientnet_b0', num_classes=NUM_CLASSES)
tot_p, tr_p = count_params(sample_model)
print(f'EfficientNet-B0 initialized: {tot_p:,} total parameters ({tr_p:,} trainable).')


---
## Cell (d): Stage 1 — Freeze Backbone & Train Head
Freezes all convolutional feature extractors and trains only the newly initialized classifier head for **5 epochs** at `lr = 1e-3`.


In [ ]:
# CELL (d): Stage 1 Helper - Freeze Backbone
def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    head = model.get_classifier()
    if isinstance(head, nn.Module):
        for param in head.parameters():
            param.requires_grad = True
    else:
        for name, param in model.named_parameters():
            if any(k in name for k in ['classifier', 'head', 'fc']):
                param.requires_grad = True

# Verify Stage 1 parameter freezing
freeze_backbone(sample_model)
tot_p, tr_p = count_params(sample_model)
print(f'Stage 1 Freezing verified: {tr_p:,} trainable out of {tot_p:,} ({tr_p/tot_p*100:.2f}%)')


---
## Cell (e): Stage 2 — Unfreeze Top Blocks & Fine-tune
Unfreezes the top layers (`blocks.5`, `blocks.6`, `conv_head`) and trains for **10 epochs** at `lr = 1e-4` with Cosine Annealing decay.


In [ ]:
# CELL (e): Stage 2 Helper - Unfreeze Top Blocks
def unfreeze_top_blocks(model, model_name='efficientnet_b0'):
    if 'efficientnet' in model_name:
        for name, param in model.named_parameters():
            if any(b in name for b in ['blocks.5', 'blocks.6', 'conv_head', 'bn2', 'classifier']):
                param.requires_grad = True
            else:
                param.requires_grad = False
    elif 'mobilenet' in model_name:
        for name, param in model.named_parameters():
            if any(b in name for b in ['blocks.4', 'blocks.5', 'conv_head', 'classifier']):
                param.requires_grad = True
            else:
                param.requires_grad = False

# Verify Stage 2 unfreezing
unfreeze_top_blocks(sample_model, 'efficientnet_b0')
tot_p, tr_p = count_params(sample_model)
print(f'Stage 2 Unfreezing verified: {tr_p:,} trainable out of {tot_p:,} ({tr_p/tot_p*100:.2f}%)')


---
## Cell (f): Full Two-Stage Training Loop
- **Label Smoothing (0.1):** Prevents overconfidence on mislabeled or ambiguous photos.
- **Class Weights:** Inverse frequency weighting balancing minority classes.
- **Mixed Precision (AMP):** Uses `torch.amp.autocast('cuda')` and `GradScaler` for 2x speedup on T4 GPU.
- **Early Stopping:** Stops if validation loss does not decrease after 5 epochs.
- **Crash Recovery:** Saves `last_checkpoint.pth` every epoch to resume seamlessly if Colab resets.


In [ ]:
# CELL (f): Complete Two-Stage Training Function
def compute_class_weights(dataset, device):
    counts = np.bincount([t for _, t in dataset.samples])
    weights = len(dataset) / (len(counts) * counts.astype(np.float32))
    return torch.tensor(weights, dtype=torch.float32, device=device)

def train_model(
    model_name='efficientnet_b0',
    stage1_epochs=5,
    stage2_epochs=10,
    stage1_lr=1e-3,
    stage2_lr=1e-4,
    label_smoothing=0.1,
    use_class_weights=True,
    resume=True
):
    print(f'\n[START] Training {model_name} on {device}')
    best_ckpt = CHECKPOINT_DIR / f'{model_name}_best.pth'
    last_ckpt = CHECKPOINT_DIR / f'{model_name}_last.pth'
    metrics_csv = CHECKPOINT_DIR / f'metrics_{model_name}.csv'

    # Loss function with optional class weights & label smoothing
    weights = compute_class_weights(train_dataset, device) if use_class_weights else None
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)

    model = build_model(model_name, num_classes=NUM_CLASSES).to(device)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    best_val_loss = float('inf')
    best_val_acc = 0.0
    start_epoch = 1
    history = []

    if resume and last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt['epoch'] + 1
        best_val_loss = ckpt['best_val_loss']
        best_val_acc = ckpt.get('best_val_acc', 0.0)
        print(f'[RESUME] Recovered from epoch {start_epoch-1} (Best Val Loss: {best_val_loss:.4f})')

    total_epochs = stage1_epochs + stage2_epochs

    # --- STAGE 1: Head Only ---
    if start_epoch <= stage1_epochs:
        print(f'\n=== STAGE 1: Head Only ({stage1_epochs} epochs, lr={stage1_lr}) ===')
        freeze_backbone(model)
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=stage1_lr, weight_decay=1e-4)

        for epoch in range(start_epoch, stage1_epochs + 1):
            model.train()
            tr_loss, correct, total = 0.0, 0, 0
            for imgs, lbls in train_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad(set_to_none=True)
                if device.type == 'cuda':
                    with torch.amp.autocast('cuda', dtype=torch.float16):
                        outputs = model(imgs)
                        loss = criterion(outputs, lbls)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(imgs)
                    loss = criterion(outputs, lbls)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                tr_loss += loss.item() * imgs.size(0)
                correct += (outputs.argmax(1) == lbls).sum().item()
                total += lbls.size(0)
            tr_loss, tr_acc = tr_loss / total, correct / total

            # Validation
            model.eval()
            v_loss, v_corr, v_tot = 0.0, 0, 0
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    outputs = model(imgs)
                    loss = criterion(outputs, lbls)
                    v_loss += loss.item() * imgs.size(0)
                    v_corr += (outputs.argmax(1) == lbls).sum().item()
                    v_tot += lbls.size(0)
            v_loss, v_acc = v_loss / v_tot, v_corr / v_tot
            print(f'Epoch {epoch:02d}/{total_epochs:02d} [Stage 1] | Train Loss: {tr_loss:.4f} Acc: {tr_acc*100:.2f}% | Val Loss: {v_loss:.4f} Acc: {v_acc*100:.2f}%')
            history.append({'epoch': epoch, 'stage': 1, 'train_loss': tr_loss, 'train_acc': tr_acc, 'val_loss': v_loss, 'val_acc': v_acc})

            if v_loss < best_val_loss:
                best_val_loss, best_val_acc = v_loss, v_acc
                torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_loss': v_loss, 'val_acc': v_acc, 'classes': CLASS_NAMES}, best_ckpt)
            torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'best_val_loss': best_val_loss, 'best_val_acc': best_val_acc}, last_ckpt)

    # --- STAGE 2: Fine-Tuning Top Blocks ---
    stage2_start = max(start_epoch, stage1_epochs + 1)
    if stage2_start <= total_epochs:
        print(f'\n=== STAGE 2: Top Blocks ({stage2_epochs} epochs, lr={stage2_lr} with Cosine LR) ===')
        unfreeze_top_blocks(model, model_name)
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=stage2_lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=stage2_epochs, eta_min=1e-6)
        patience = 0

        for epoch in range(stage2_start, total_epochs + 1):
            model.train()
            tr_loss, correct, total = 0.0, 0, 0
            for imgs, lbls in train_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad(set_to_none=True)
                if device.type == 'cuda':
                    with torch.amp.autocast('cuda', dtype=torch.float16):
                        outputs = model(imgs)
                        loss = criterion(outputs, lbls)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(imgs)
                    loss = criterion(outputs, lbls)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                tr_loss += loss.item() * imgs.size(0)
                correct += (outputs.argmax(1) == lbls).sum().item()
                total += lbls.size(0)
            scheduler.step()
            tr_loss, tr_acc = tr_loss / total, correct / total

            # Validation
            model.eval()
            v_loss, v_corr, v_tot = 0.0, 0, 0
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    outputs = model(imgs)
                    loss = criterion(outputs, lbls)
                    v_loss += loss.item() * imgs.size(0)
                    v_corr += (outputs.argmax(1) == lbls).sum().item()
                    v_tot += lbls.size(0)
            v_loss, v_acc = v_loss / v_tot, v_corr / v_tot
            print(f'Epoch {epoch:02d}/{total_epochs:02d} [Stage 2] | Train Loss: {tr_loss:.4f} Acc: {tr_acc*100:.2f}% | Val Loss: {v_loss:.4f} Acc: {v_acc*100:.2f}% | lr: {scheduler.get_last_lr()[0]:.1e}')
            history.append({'epoch': epoch, 'stage': 2, 'train_loss': tr_loss, 'train_acc': tr_acc, 'val_loss': v_loss, 'val_acc': v_acc})

            if v_loss < best_val_loss:
                best_val_loss, best_val_acc = v_loss, v_acc
                patience = 0
                torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_loss': v_loss, 'val_acc': v_acc, 'classes': CLASS_NAMES}, best_ckpt)
                print(f'  --> Best model checkpoint updated! (Val Loss: {v_loss:.4f}, Val Acc: {v_acc*100:.2f}%)')
            else:
                patience += 1
                if patience >= 5:
                    print(f'[EARLY STOP] No validation improvement for 5 epochs. Halting.')
                    break
            torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'best_val_loss': best_val_loss, 'best_val_acc': best_val_acc}, last_ckpt)

    pd.DataFrame(history).to_csv(metrics_csv, index=False)
    print(f'[COMPLETE] {model_name} finished. Best Val Acc: {best_val_acc*100:.2f}%. Saved to {best_ckpt}')
    return best_ckpt, metrics_csv

# Train Main EfficientNet-B0 Model
eff_ckpt, eff_csv = train_model('efficientnet_b0', stage1_epochs=5, stage2_epochs=10)


---
## Cell (g): Loss and Accuracy Learning Curves
Visualizes train vs validation loss and accuracy to verify model convergence and inspect generalization gaps.


In [ ]:
# CELL (g): Plotting Curves
def plot_training_curves(metrics_csv, title='EfficientNet-B0'):
    df = pd.read_csv(metrics_csv)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Loss Curve
    ax1.plot(df['epoch'], df['train_loss'], label='Train Loss', marker='o')
    ax1.plot(df['epoch'], df['val_loss'], label='Val Loss', marker='s')
    ax1.axvline(x=5.5, color='gray', linestyle='--', label='Stage 2 Fine-Tuning Start')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} - Loss Curve')
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    # Accuracy Curve
    ax2.plot(df['epoch'], df['train_acc'] * 100, label='Train Acc', marker='o')
    ax2.plot(df['epoch'], df['val_acc'] * 100, label='Val Acc', marker='s')
    ax2.axvline(x=5.5, color='gray', linestyle='--', label='Stage 2 Fine-Tuning Start')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'{title} - Accuracy Curve')
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    plt.tight_layout()
    save_path = CHECKPOINT_DIR / f'curve_{title.lower().replace(" ", "_")}.png'
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f'Curves saved to {save_path}')

plot_training_curves(eff_csv, 'EfficientNet-B0')


---
## MobileNetV3-Small Baseline Model
Trains MobileNetV3-Small using the identical pipeline to evaluate lightweight architecture trade-offs in accuracy vs model complexity.


In [ ]:
# Train MobileNetV3-Small Baseline
mob_ckpt, mob_csv = train_model('mobilenetv3_small_100', stage1_epochs=5, stage2_epochs=10)
plot_training_curves(mob_csv, 'MobileNetV3-Small')


---
## 🧪 Ablation Plan
To scientifically justify our architectural choices in the project report, we compare 4 configurations on the validation split:

| Configuration | Backbone Status | Data Augmentation | Label Smoothing | Purpose | Fill In: Val Acc (%) |
|---|---|---|---|---|---|
| **M1: Baseline Head-Only** | Frozen (Stage 1 only) | Yes | 0.1 | Demonstrates value of fine-tuning | FILL_IN |
| **M2: No Augmentation** | Full 2-Stage | **No** | 0.1 | Measures impact of rotation/flip/jitter | FILL_IN |
| **M3: No Label Smoothing** | Full 2-Stage | Yes | **0.0** | Measures impact of regularization on overconfidence | FILL_IN |
| **M4: Proposed Pipeline** | Full 2-Stage | Yes | 0.1 | Full proposed model | FILL_IN |
